In [286]:
import pandas as pd
pd.set_option("display.max_columns", None)

### Lecture des fichier csv

In [287]:
BASE = "datalake/transfermarkt/"

In [ ]:
player_profiles = pd.read_csv(BASE + "player_profiles/player_profiles.csv")
player_market_values = pd.read_csv(BASE + "player_market_value/player_market_value.csv")
player_latest_market_values = pd.read_csv(BASE + "player_latest_market_value/player_latest_market_value.csv")
player_performances = pd.read_csv(BASE + "player_performances/player_performances.csv")
player_transfer_histories = pd.read_csv(BASE + "transfer_history/transfer_history.csv")
player_injury_histories = pd.read_csv(BASE + "player_injuries/player_injuries.csv")
player_national_perf = pd.read_csv(BASE + "player_national_performances/player_national_performances.csv")
player_teammates = pd.read_csv(BASE + "player_teammates_played_with/player_teammates_played_with.csv")
teams = pd.read_csv(BASE + "team_details/team_details.csv")
teams_children = pd.read_csv(BASE + "team_children/team_children.csv")
teams_seasons = pd.read_csv(BASE + "team_competitions_seasons/team_competitions_seasons.csv")

/var/folders/x8/35lmzvhd5wd_s5blbbddvzdr0000gn/T/ipykernel_68961/4141833267.py:1: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  player_profiles = pd.read_csv(BASE + "player_profiles/player_profiles.csv")


## Traitement des fichier
### Traitement sur les profils de joueurs

In [ ]:
player_profiles=player_profiles[player_profiles['date_of_death'].isna()]
player_profiles=player_profiles[player_profiles['current_club_name']!='Retired']
player_profiles["second_nationality"] = (player_profiles["citizenship"]
                                         .str.split("  ")   # séparation sur double espace
                                         .str[1])
player_profiles["first_nationality"] = (player_profiles["citizenship"]
                                         .str.split("  ")   # séparation sur double espace
                                         .str[0])
player_profiles = player_profiles.drop(['player_image_url','citizenship','date_of_death','name_in_home_country','social_media_url','place_of_birth','country_of_birth','height','is_eu','outfitter','player_agent_id','player_agent_name','contract_option','second_club_url','third_club_url','fourth_club_url'], axis=1)
player_profiles=player_profiles[['player_id', 'player_slug', 'player_name', 'date_of_birth',  'first_nationality','second_nationality','position',
       'main_position', 'foot', 'current_club_id', 'current_club_name',
       'joined', 'contract_expires', 'date_of_last_contract_extension',
       'on_loan_from_club_id', 'on_loan_from_club_name',
       'contract_there_expires', 'second_club_name', 'third_club_name',
       'fourth_club_name']]

### Traitement sur les performances de joueurs

In [ ]:
player_performances=player_performances[player_performances['season_name'].str.contains('/')]
player_performances=player_performances.groupby(['season_name','player_id','team_id','team_name']).agg({'nb_in_group':'sum','goals':'sum','assists':'sum','yellow_cards':'sum','direct_red_cards':'sum','goals_conceded':'sum','clean_sheets':'sum'}).reset_index()
player_performances = player_performances.sort_values('goals',ascending=False).drop_duplicates(subset=["player_id", "season_name"], keep="first")
player_performances.sort_values('player_id',ascending=False)
def convert_season(s):
    first = int(s.split("/")[0])
    if first < 27:
        return 2000 + first
    else:
        return 1900 + first
player_performances["year"] = player_performances["season_name"].apply(convert_season)

### Traitement sur les valeurs de joueurs

In [300]:
player_market_values["year"] = pd.to_datetime(player_market_values["date_unix"]).dt.year
player_market_values=player_market_values.sort_values("value", ascending=False).drop_duplicates(subset=["player_id", "year"],keep="first")

In [292]:
print("player_profiles :", player_profiles.shape)
print("player_market_values :", player_market_values.shape)
print("player_performances :", player_performances.shape)

player_profiles : (69022, 20)
player_market_values : (901429, 4)
player_performances : (672113, 12)


In [303]:
Dataset=pd.merge(player_market_values,player_profiles,
    on=["player_id"],
    how="left")
Dataset=pd.merge(Dataset,player_performances,
    on=["player_id","year"],
    how="left")
Dataset['age']=(Dataset['year'] - pd.to_datetime(Dataset['date_of_birth']).dt.year)

In [308]:
Dataset.head()

,player_id,date_unix,value,year,player_slug,player_name,date_of_birth,first_nationality,second_nationality,position,main_position,foot,current_club_id,current_club_name,joined,contract_expires,date_of_last_contract_extension,on_loan_from_club_id,on_loan_from_club_name,contract_there_expires,second_club_name,third_club_name,fourth_club_name,season_name,team_id,team_name,nb_in_group,goals,assists,yellow_cards,direct_red_cards,goals_conceded,clean_sheets,age
0,371998,2024-12-26,200000000.0,2024,vinicius-junior,Vinicius Junior (371998),2000-07-12,Brazil,Spain,Attack - Left Winger,Attack,right,418.0,Real Madrid,2018-07-12,2027-06-30,2023-10-31,NaN,NaN,NaN,NaN,NaN,NaN,24/25,418.0,Real Madrid,52.0,21.0,18.0,15.0,1.0,0.0,0.0,24.0
1,342229,2018-12-16,200000000.0,2018,kylian-mbappe,Kylian Mbappé (342229),1998-12-20,France,Cameroon,Attack - Centre-Forward,Attack,right,418.0,Real Madrid,2024-07-01,2029-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18/19,583.0,Paris Saint-Germain,44.0,39.0,18.0,6.0,2.0,0.0,0.0,20.0
2,418560,2024-09-30,200000000.0,2024,erling-haaland,Erling Haaland (418560),2000-07-21,Norway,NaN,Attack - Centre-Forward,Attack,left,281.0,Manchester City,2022-07-01,2034-06-30,2025-01-17,NaN,NaN,NaN,NaN,NaN,NaN,24/25,281.0,Manchester City,48.0,31.0,4.0,2.0,0.0,0.0,0.0,24.0
3,342229,2019-06-02,200000000.0,2019,kylian-mbappe,Kylian Mbappé (342229),1998-12-20,France,Cameroon,Attack - Centre-Forward,Attack,right,418.0,Real Madrid,2024-07-01,2029-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19/20,583.0,Paris Saint-Germain,39.0,30.0,18.0,4.0,0.0,0.0,0.0,21.0
4,937958,2025-06-08,200000000.0,2025,lamine-yamal,Lamine Yamal (937958),2007-07-13,Spain,Equatorial Guinea,Attack - Right Winger,Attack,left,131.0,FC Barcelona,2023-07-01,2031-06-30,2025-05-27,NaN,NaN,NaN,NaN,NaN,NaN,25/26,131.0,FC Barcelona,3.0,2.0,3.0,0.0,0.0,0.0,0.0,18.0


## Detection des biais

In [309]:
import pandas as pd

def detect_bias(df):
    report = []

    # 1. Missing data bias
    missing = df.isna().mean().sort_values(ascending=False)
    high_missing = missing[missing > 0.20]  # threshold 20%
    if not high_missing.empty:
        report.append("⚠️ **Biais de données manquantes :**\n" +
                      "\n".join([f"- {col}: {pct:.1%} manquant" for col, pct in high_missing.items()]))

    # 2. Distribution bias (nationalities)
    if 'first_nationality' in df.columns:
        nat_counts = df['first_nationality'].value_counts(normalize=True)
        if nat_counts.max() > 0.50:
            report.append(f"⚠️ **Biais de nationalité :** '{nat_counts.idxmax()}' représente {nat_counts.max():.1%} du dataset.")

    # 3. Position bias
    if 'main_position' in df.columns:
        pos_counts = df['main_position'].value_counts(normalize=True)
        if pos_counts.max() > 0.50:
            report.append(f"⚠️ **Biais de position :** La position '{pos_counts.idxmax()}' domine ({pos_counts.max():.1%}).")

    # 4. Age distribution bias
    if 'age' in df.columns:
        if df['age'].median() < 22 or df['age'].median() > 30:
            report.append("⚠️ **Biais d'âge :** distribution très jeune ou très âgée.")

    # 5. Value bias by nationality
    if 'value' in df.columns and 'first_nationality' in df.columns:
        nat_value = df.groupby("first_nationality")['value'].mean()
        ratio = nat_value.max() / nat_value.min()
        if ratio > 3:
            report.append("⚠️ **Biais de valeur économique :** forte différence de valeur moyenne entre nationalités.")

    # 6. Performance bias by club
    if 'goals' in df.columns and 'current_club_name' in df.columns:
        club_goals = df.groupby("current_club_name")['goals'].mean()
        if club_goals.max() > 2 * club_goals.mean():
            report.append("⚠️ **Biais sportif :** certains clubs sur-performent fortement (goals).")

    # 7. Footedness bias (right/left)
    if 'foot' in df.columns:
        foot_counts = df['foot'].value_counts(normalize=True)
        if foot_counts.max() > 0.80:
            report.append(f"⚠️ **Biais pied dominant :** '{foot_counts.idxmax()}' représente {foot_counts.max():.1%}.")

    # 8. Season bias
    if 'season_name' in df.columns:
        season_counts = df['season_name'].value_counts(normalize=True)
        if season_counts.max() > 0.50:
            report.append(f"⚠️ **Biais temporel :** la saison '{season_counts.idxmax()}' domine ({season_counts.max():.1%}).")

    # Final report
    if not report:
        return "Aucun biais significatif détecté ✔️"
    else:
        return "\n\n".join(report)


print(detect_bias(Dataset))


⚠️ **Biais de données manquantes :**
- fourth_club_name: 100.0% manquant
- third_club_name: 100.0% manquant
- second_club_name: 99.7% manquant
- contract_there_expires: 97.5% manquant
- on_loan_from_club_id: 96.8% manquant
- on_loan_from_club_name: 96.8% manquant
- date_of_last_contract_extension: 88.5% manquant
- second_nationality: 82.0% manquant
- contract_expires: 51.8% manquant
- foot: 32.7% manquant
- player_name: 27.6% manquant
- joined: 27.4% manquant
- age: 27.3% manquant
- date_of_birth: 27.3% manquant
- first_nationality: 27.2% manquant
- current_club_name: 27.2% manquant
- main_position: 27.2% manquant
- position: 27.2% manquant
- player_slug: 27.2% manquant
- current_club_id: 27.2% manquant
- goals: 22.8% manquant
- clean_sheets: 22.8% manquant
- goals_conceded: 22.8% manquant
- direct_red_cards: 22.8% manquant
- yellow_cards: 22.8% manquant
- assists: 22.8% manquant
- season_name: 22.8% manquant
- nb_in_group: 22.8% manquant
- team_name: 22.8% manquant
- team_id: 22.8% ma